<a href="https://colab.research.google.com/github/akriti404/Deep-Learning-Lab/blob/main/lab6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader

class InceptionBlock(nn.Module):

    def __init__(self, in_channels, n1x1, n3x3_reduce, n3x3, n5x5_reduce, n5x5, pool_proj):

        super(InceptionBlock, self).__init__()
        self.branch1 = nn.Sequential(

            nn.Conv2d(in_channels, n1x1, kernel_size=1),

            nn.BatchNorm2d(n1x1),

            nn.ReLU(True)

        )
        self.branch2 = nn.Sequential(

            nn.Conv2d(in_channels, n3x3_reduce, kernel_size=1),

            nn.ReLU(True),

            nn.Conv2d(n3x3_reduce, n3x3, kernel_size=3, padding=1),

            nn.BatchNorm2d(n3x3),

            nn.ReLU(True)

        )

        self.branch3 = nn.Sequential(

            nn.Conv2d(in_channels, n5x5_reduce, kernel_size=1),

            nn.ReLU(True),

            nn.Conv2d(n5x5_reduce, n5x5, kernel_size=5, padding=2),

            nn.BatchNorm2d(n5x5),

            nn.ReLU(True)

        )

        self.branch4 = nn.Sequential(

            nn.MaxPool2d(kernel_size=3, stride=1, padding=1),

            nn.Conv2d(in_channels, pool_proj, kernel_size=1),

            nn.BatchNorm2d(pool_proj),

            nn.ReLU(True)

        )



    def forward(self, x):

        out1 = self.branch1(x)

        out2 = self.branch2(x)

        out3 = self.branch3(x)

        out4 = self.branch4(x)

        return torch.cat([out1, out2, out3, out4], dim=1)

class MiniInceptionNet(nn.Module):

    def __init__(self, num_classes=10):

        super(MiniInceptionNet, self).__init__()

        self.stem = nn.Sequential(

            nn.Conv2d(3, 64, kernel_size=3, padding=1),

            nn.BatchNorm2d(64),

            nn.ReLU(True)

        )

        self.inception3a = InceptionBlock(64, 32, 32, 64, 8, 16, 16)

  # Output channels: 32+64+16+16 = 128

        self.inception3b = InceptionBlock(128, 64, 32, 64, 16, 32, 32)

 # Output channels: 64+64+32+32 = 192



        self.maxpool = nn.MaxPool2d(kernel_size=3, stride=2, padding=1)



        self.inception4a = InceptionBlock(192, 64, 48, 96, 16, 32, 32)  # Output channels: 64+96+32+32 = 224



        # Classifier

        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))

        self.dropout = nn.Dropout(0.4)

        self.fc = nn.Linear(224, num_classes)



    def forward(self, x):

        x = self.stem(x)

        x = self.inception3a(x)

        x = self.inception3b(x)

        x = self.maxpool(x)

        x = self.inception4a(x)

        x = self.global_pool(x)

        x = torch.flatten(x, 1)

        x = self.dropout(x)

        x = self.fc(x)

        return x



def main():

    # Set device

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    print(f"Using device: {device}")



    # Transforms for CIFAR-10

    transform_train = transforms.Compose([

        transforms.RandomHorizontalFlip(),

        transforms.ToTensor(),

        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))

    ])



    transform_test = transforms.Compose([

        transforms.ToTensor(),

        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))

    ])



    # Datasets and Loaders

    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform_train)

    train_loader = DataLoader(trainset, batch_size=64, shuffle=True, num_workers=2)



    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform_test)

    test_loader = DataLoader(testset, batch_size=64, shuffle=False, num_workers=2)



    # Initialize Model, Loss, and Optimizer

    model = MiniInceptionNet(num_classes=10).to(device)

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.Adam(model.parameters(), lr=0.001)



    # Training Loop (3 Epochs for quick demonstration)

    epochs = 3

    print("\nStarting Training...")

    for epoch in range(epochs):

        model.train()

        running_loss = 0.0

        for i, (images, labels) in enumerate(train_loader):

            images, labels = images.to(device), labels.to(device)



            optimizer.zero_grad()

            outputs = model(images)

            loss = criterion(outputs, labels)

            loss.backward()

            optimizer.step()



            running_loss += loss.item()

            if (i + 1) % 200 == 0:

                print(f"Epoch [{epoch+1}/{epochs}], Step [{i+1}/{len(train_loader)}], Loss: {running_loss/200:.4f}")

                running_loss = 0.0



    # Evaluation Phase

    model.eval()

    correct = 0

    total = 0

    with torch.no_grad():

        for images, labels in test_loader:

            images, labels = images.to(device), labels.to(device)

            outputs = model(images)

            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)

            correct += (predicted == labels).sum().item()



    print(f"\nAccuracy on the 10,000 test images: {100 * correct / total:.2f}%")



if __name__ == "__main__":

    main()

Using device: cuda


100%|██████████| 170M/170M [41:04<00:00, 69.2kB/s]



Starting Training...
Epoch [1/3], Step [200/782], Loss: 1.6792
Epoch [1/3], Step [400/782], Loss: 1.3705
Epoch [1/3], Step [600/782], Loss: 1.2317
Epoch [2/3], Step [200/782], Loss: 1.1049
Epoch [2/3], Step [400/782], Loss: 1.0606
Epoch [2/3], Step [600/782], Loss: 1.0143
Epoch [3/3], Step [200/782], Loss: 0.9592
Epoch [3/3], Step [400/782], Loss: 0.9349
Epoch [3/3], Step [600/782], Loss: 0.9206

Accuracy on the 10,000 test images: 65.94%
